# Uponor plots

## Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import plotly.express as px
import plotly.graph_objects as go
from src.utils.uponor_plots import *
import pandas as pd
import numpy as np
from pprint import pprint
from copy import deepcopy
from datetime import datetime

## Load data

In [12]:
training_progress = {
    
    'digital_twin': pd.read_csv('data/Eplus-PPO-digital-twin-Example_2026-02-26_10:01-res1/progress.csv'),
    
    
}

evaluation_obs = {
    'digital_twin': pd.read_csv('data/Eplus-PPO-digital-twin-Example_2026-02-26_10:01-res1/episode-10/monitor/observations.csv'),
}

evaluation_actions = {
    'digital_twin': pd.read_csv('data/Eplus-PPO-digital-twin-Example_2026-02-26_10:01-res1/episode-10/monitor/simulated_actions.csv'),
}

evaluation_actions['digital_twin'].rename(columns={'flow_rate_f0_living-kitchen': 'flow_rate_f0_living-kitchen_action',
                                   'flow_rate_f0_bathroom-lobby': 'flow_rate_f0_bathroom-lobby_action',
                                   'flow_rate_f1_bedroom1': 'flow_rate_f1_bedroom1_action',
                                   'flow_rate_f1_bedroom2': 'flow_rate_f1_bedroom2_action',
                                   'flow_rate_f1_bedroom3': 'flow_rate_f1_bedroom3_action',
                                   'flow_rate_f1_bathroom-corridor': 'flow_rate_f1_bathroom-corridor_action',
                                   'flow_rate_f1_bathroom-dressing': 'flow_rate_f1_bathroom-dressing_action',
                                   'water_temperature': 'water_temperature_action'}, inplace=True)

evaluation_infos = {
    'digital_twin': pd.read_csv('data/Eplus-PPO-digital-twin-Example_2026-02-26_10:01-res1/episode-10/monitor/infos.csv'),
}

#Concatenate the actions and observations
unified = {
    key: pd.concat([
        evaluation_obs[key], 
        evaluation_actions[key], 
        evaluation_infos[key]
    ], axis=1)
    for key in evaluation_obs.keys()
}

evaluation_progress = {
    'digital_twin': pd.read_csv('data/Eplus-PPO-digital-twin-Example_2026-02-26_10:01-res1/progress.csv'),
}

### Preprocess

Añadimos la columna datetime y filtramos dentro el rango específico que queremos analizar

In [13]:
for key,df in unified.items():
    unified[key] = add_datetime_column(unified[key])
    unified[key] = filer_interval(unified[key], '2022-11-15 00:00:00', '2023-03-15 23:55:00')
    #Delete rows in May using datetime column
    unified[key] = unified[key][unified[key]['datetime'].dt.month != 5]

### Variables

In [14]:
df_num = len(unified)
zone_names=[
    'Living Kitchen room', 
    'Bathroom lobby',
    'Bedroom1', 
    'Bedroom2', 
    'Bedroom3',
    'Bathroom corridor',
    'Bathroom dressing'
]
zone_size = len(zone_names)
temperature_variables = [
    'air_temperature_f0_living-kitchen',
    'air_temperature_f0_bathroom-lobby',
    'air_temperature_f1_bedroom1',
    'air_temperature_f1_bedroom2',
    'air_temperature_f1_bedroom3',
    'air_temperature_f1_bathroom-corridor',
    'air_temperature_f1_bathroom-dressing'
]
flow_variables = [
    'flow_rate_f0_living-kitchen',
    'flow_rate_f0_bathroom-lobby',
    'flow_rate_f1_bedroom1',
    'flow_rate_f1_bedroom2',
    'flow_rate_f1_bedroom3',
    'flow_rate_f1_bathroom-corridor',
    'flow_rate_f1_bathroom-dressing'
]
energy_variable = 'heat_source_electricity_rate'
colors = [
    '#1ABC9C',  # Turquesa
    '#3498DB',  # Azul claro
    '#9B59B6',  # Púrpura
    '#E74C3C',  # Rojo
    '#F1C40F',  # Amarillo
    '#2ECC71',  # Verde
    '#E67E22',  # Naranja
    '#95A5A6',  # Gris claro
    '#34495E',  # Azul oscuro grisáceo
    '#D35400',   # Naranja oscuro
    '#2C3E50',   # Azul oscuro grisáceo
    '#7F8C8D',   # Gris medio
    '#BDC3C7',   # Gris claro
    '#ECF0F1',   # Gris muy claro
    '#34485E'   # Azul oscuro grisáceo
    '#EC49F1',   # Gris muy claro
    '#EC29F1',   # Gris muy claro
    '#EC21F1'   # Gris muy claro

    
]
action_variables = [
    'water_temperature',
    'flow_rate_f0_living-kitchen_action',
    'flow_rate_f0_bathroom-lobby_action',
    'flow_rate_f1_bedroom1_action',
    'flow_rate_f1_bedroom2_action',
    'flow_rate_f1_bedroom3_action',
    'flow_rate_f1_bathroom-corridor_action',
    'flow_rate_f1_bathroom-dressing_action'
]

### Recalculate Means

In [15]:
mean_temp_violation_dict={}
mean_energy_consumption_dict={}
mean_heat_transfer_rate_dict={}

for key, df in unified.items():
    mean_temp_violation_dict[key] = mean_variable(df, variable='total_temperature_violation')
    mean_energy_consumption_dict[key] = mean_variable(df, variable='total_power_demand')
    mean_heat_transfer_rate_dict[key] = mean_variable(df, variable='heat_source_load_side_heat_transfer_rate')


## Results

### Training Progress

In [16]:
fig = plot_dfs_line(df_dict= training_progress, variable_name ='mean_reward', colors = colors[:df_num])
fig.update_layout(title='Training Progress', xaxis_title='Episode', yaxis_title='Mean Reward',
                  legend=dict(
                    yanchor="bottom",
                    y=0.01,
                    xanchor="right",
                    x=0.99,
                    orientation="v"
                ))
#fig.write_image("output_images/training_progress.png")
fig.show()

### Temperatures in one episode

In [17]:
for key,df in unified.items():
    fig = plot_temperatures(df = unified[key], 
                            variables=temperature_variables, 
                            names=zone_names,   
                            colors=colors)

    fig.update_layout(title=key, xaxis_title='', yaxis_title='Temperature (°C)')
    fig.show()
    #fig.write_image("output_images/"+key+"_air_temperatures.png")

### Temperature vs On/Off flow

In [18]:
for key,df in unified.items():
    if key=='short_cycling_2T':
        fig = plot_control(df = unified[key], 
                                temperature_variables=temperature_variables,
                                flow_variables=flow_variables,
                                names=[f'Temp {z}' for z in zone_names] + [f'Flow {z}' for z in zone_names],   
                                colors=colors[:10])

        #fig.update_layout(title=key, xaxis_title='', yaxis_title='Temperature (°C)')
        fig.show()

### Means (heat transfer, power consumption, violation temp)

In [19]:
fig = plot_bar(mean_heat_transfer_rate_dict, bar_colors=colors[:df_num])
fig.update_layout(title='Heat Transfer Rate',
                  xaxis_title='',
                  yaxis_title='')
fig.show()
#fig.write_image("output_images/mean_heat_transfer_rate.png")

In [20]:
fig = plot_bar(mean_energy_consumption_dict, bar_colors=colors[:df_num])
fig.update_layout(title='Comparative power demand',
                  xaxis_title='',
                  yaxis_title='Mean power consumption (W)')
fig.show()
#fig.write_image("/workspaces/python-plot-env/outputs/mean_power_demand.png")

In [21]:
fig = plot_bar(mean_temp_violation_dict, bar_colors=colors[:df_num])
fig.update_layout(title='Comparative mean temperature violation',
                  xaxis_title='',
                  yaxis_title='Mean Temperature Violation (ºC)')

#fig.write_image("/workspaces/python-plot-env/outputs/mean_temp_violations.png")
fig.show()

### Means by month

In [22]:
fig = plot_dfs_line_grouped_by_month(dfs = list(unified.values()),
                                     names=list(unified.keys()),
                                     colors = colors[:df_num],
                                     variable='heat_source_electricity_rate')
fig.update_layout(title='Mean Power Consumption',
                  xaxis_title=None,
                  yaxis_title='Power demand (W)')
#fig.write_image("/workspaces/python-plot-env/outputs/month_power_demand.png")
fig.show()

In [23]:
fig = plot_dfs_line_grouped_by_month(dfs = list(unified.values()),
                                     names=list(unified.keys()),
                                     colors = colors[:df_num],
                                     variable='plr_current')
fig.update_layout(title='PLR',
                  xaxis_title=None,
                  yaxis_title=None)
#fig.write_image("/workspaces/python-plot-env/outputs/month_plr.png")
fig.show()


In [24]:
fig = plot_dfs_line_grouped_by_month(dfs = list(unified.values()),
                                     names=list(unified.keys()),
                                     colors = colors[:df_num],
                                     variable='heat_source_load_side_heat_transfer_rate')
fig.update_layout(title='Heat source Load Side Heat Transfer Rate',
                  xaxis_title=None,
                  yaxis_title=None)
#fig.write_image("/workspaces/python-plot-env/outputs/month_heat_transfer_rate.png")
fig.show()

### Energy savings

In [25]:
# Calculate the mean energy consumption savings (%) against the baseline
mean_energy_consumption_savings_dict = {}
reference_baseline = 'case2_baseline'
for key, value in mean_energy_consumption_dict.items():  
    mean_energy_consumption_savings_dict[key] = 100*(mean_energy_consumption_dict[reference_baseline] - value)/mean_energy_consumption_dict[reference_baseline]

del mean_energy_consumption_savings_dict[reference_baseline]

# Plot the mean energy consumption savings
fig = plot_bar(mean_energy_consumption_savings_dict, bar_colors=colors[1:df_num])
fig.update_layout(title=f'Comparative mean energy consumption savings vs {reference_baseline}',
                  xaxis_title='',
                  yaxis_title='Mean energy consumption savings (%)')
#fig.write_image("/workspaces/python-plot-env/outputs/mean_savings.png")
fig.show()

KeyError: 'case2_baseline'

### Energy savings (%) by month

In [17]:
combination_size = 1*3
fig = plot_energy_savings(data=unified,
                          names_reference=['case2_baseline'],
                          names_comparison=['first_75', 'first_50', 'second_50'],
                          variable='heat_source_electricity_rate',
                          colors = colors[1:combination_size+1],)
fig.write_image("/workspaces/python-plot-env/outputs/month_energy_savings.png")

### Action distribution

In [21]:
for action_variable in action_variables:
    fig = plot_action_distribution(dfs=list(unified.values()), 
                                   names=list(unified.keys()), 
                                   variable=action_variable,
                                   colors = colors[:df_num])
    fig.update_layout(
        title = f'{action_variable} Distribution',
        yaxis_title = '',
        showlegend=False,
    )
    #fig.write_image(f"/workspaces/python-plot-env/outputs/distribution_{action_variable}.png")
    fig.show()

### BoxPlot

In [19]:
fig = plot_dfs_boxplot(dfs = evaluation_progress.values(), 
                       names = evaluation_progress.keys(), 
                       variable = 'comfort_violation_time(%)',
                       colors = colors[:df_num])
fig.update_layout(title='Comfort violation time (% of episode)',
                  yaxis_title='')
fig.write_image("/workspaces/python-plot-env/outputs/boxplot_comfort_violation_time.png")

In [20]:
fig = plot_dfs_boxplot(dfs = evaluation_progress.values(), 
                       names = evaluation_progress.keys(), 
                       variable='mean_temperature_violation', 
                       colors = colors[:df_num])
fig.update_layout(title='Mean temperature violation',
                  yaxis_title='Temperature violation (ºC)')
fig.write_image("/workspaces/python-plot-env/outputs/boxplot_temperature_violation.png")

In [21]:
fig = plot_dfs_boxplot(dfs = evaluation_progress.values(), 
                       names = evaluation_progress.keys(), 
                       variable='mean_power_demand', 
                       colors = colors[:df_num])
fig.update_layout(title='Mean power demand',
                  yaxis_title='Power demand (W)')
fig.write_image("/workspaces/python-plot-env/outputs/boxplot_power_demand.png")